# 03 — Retrieval: MF-BPR

**Owner:** Person B · **Course session:** S2 (Matrix Factorization) · **25% of grade with Notebook 04**

Trains Matrix Factorization with a pairwise BPR loss from scratch (`src/models/mf_bpr.py`),
tunes on **val only**, builds a FAISS index over the learned item embeddings, and evaluates
retrieval quality against the Notebook 02 baselines.

> **Colab setup:** Runtime → Change runtime type → GPU (T4) is not required for MF-BPR (it's
> cheap — CPU is fine and saves GPU quota for Notebook 04's two-tower), but harmless either way.
> Mount Drive and point `DATA` below at your copy of Notebook 01's `data/` output if you saved
> it there instead of running Notebook 01 fresh in this session.

**Before the defense, every team member must be able to derive `bpr_loss` line-by-line and
explain why `sample_triplets` resamples a negative that collides with a user's positives** —
these are called out explicitly in `src/models/mf_bpr.py`'s docstring as defense questions.

In [1]:
!git clone https://github.com/21zaimotman-tech/amazon-recsys.git
%cd amazon-recsys
!pip install -q -r requirements.txt


Cloning into 'amazon-recsys'...
fatal: unable to access 'https://github.com/21zaimotman-tech/amazon-recsys.git/': Failed to connect to github.com port 443 after 135532 ms: Connection timed out
[Errno 2] No such file or directory: 'amazon-recsys'
/content
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [2]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # macOS: torch+faiss both bundle OpenMP and abort on double-init otherwise
import sys, time, json
from pathlib import Path
sys.path.insert(0, ".." if Path("../src").exists() else ".")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src import config as C
from src.models.mf_bpr import MFBPR, sample_triplets, export_item_embeddings
from src.retrieval.faiss_index import EmbeddingIndex
from src.eval.harness import evaluate_index, mfbpr_user_vecs
from src.data.split import ground_truth_from

plt.rcParams["figure.dpi"] = 110
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

DATA = Path("../data") if Path("../data").exists() else Path("data")
train = pd.read_parquet(DATA / "train.parquet")
val = pd.read_parquet(DATA / "val.parquet")
test = pd.read_parquet(DATA / "test.parquet")
ids = json.load(open(DATA / "id_encoders.json"))
user_ids, item_ids = ids["user_ids"], ids["item_ids"]
uid2u = {u: i for i, u in enumerate(user_ids)}       # raw user_id -> encoded u (train vocab only)
n_users, n_items = len(user_ids), len(item_ids)
print(f"train={len(train):,}  val={len(val):,}  test={len(test):,}  n_users={n_users:,}  n_items={n_items:,}")

baseline_results = pd.read_csv(DATA / "baseline_results.csv", index_col=0)
baseline_results

ModuleNotFoundError: No module named 'src'

## Id spaces, and why they matter here

Two different id spaces are in play and mixing them up silently breaks the eval:

- **Encoded ints (`u`/`i` columns)** — only used to index into the model's embedding tables
  (`model.user_emb`, `model.item_emb`). `encode_ids` (Notebook 01) reserves a trailing
  `<UNK>` row (index `n_users`/`n_items`) for a val/test id never seen in train, so embedding
  tables are built with `n_users + 1` / `n_items + 1` rows.
- **Raw string ids (`user_id`/`item_id`)** — everything from `src.eval` (`seen_items`,
  `ground_truth_from`, `evaluate`) and `EmbeddingIndex.search` operates in this space, exactly
  like the Notebook 02 baselines. `FAISS` is built with `item_ids` (the index → string list) as
  labels, so `index.search(...)` already returns raw string item ids.

So: encode only to fetch a user's embedding row, then immediately hand raw string ids to
everything eval-related below — never compare an encoded int against a raw id.

## Training loop (with loss history, for the required loss-curve plot)

Identical to `src.models.mf_bpr.train_mfbpr` — same triplet sampling, same `bpr_loss` call —
just additionally records the per-epoch mean loss for plotting. This *is* the training
function used everywhere below; there is no separate/duplicate training run.

In [ ]:
def train_mfbpr_with_history(train_df, n_users, n_items, dim, lr, reg, epochs,
                             batch_size, steps_per_epoch, device):
    """n_users/n_items here already include the +1 UNK row. `sample_triplets` still
    samples negatives only from the n_items-1 real (non-UNK) items.

    `pos`/`user_items` are built ONCE here, not inside the step loop: sample_triplets
    is called steps_per_epoch * epochs times (thousands), and rebuilding a
    groupby(\'u\')[\'i\'].agg(set).to_dict() over the full train set on every single
    call (rather than once per training run) is the actual bottleneck on a GPU
    runtime -- the GPU sits idle while the CPU redoes that pandas groupby between
    steps. Precomputing it once cuts training time by roughly steps_per_epoch x."""
    rng = np.random.default_rng(0)
    m = MFBPR(n_users, n_items, dim, reg).to(device)
    opt = torch.optim.SGD(m.parameters(), lr=lr)
    pos = train_df[["u", "i"]].values
    user_items = train_df.groupby("u")["i"].agg(set).to_dict()
    history = []
    for ep in range(epochs):
        m.train(); running = 0.0
        for _ in range(steps_per_epoch):
            u, i, j = sample_triplets(train_df, n_items - 1, rng, batch_size, pos, user_items)
            u = torch.as_tensor(u, device=device); i = torch.as_tensor(i, device=device)
            j = torch.as_tensor(j, device=device)
            opt.zero_grad(); loss = m.bpr_loss(u, i, j); loss.backward(); opt.step()
            running += loss.item()
        history.append(running / steps_per_epoch)
        print(f"epoch {ep+1}/{epochs}  loss={history[-1]:.4f}")
    return m, history


def retrieval_metrics(model, eval_df, train_df, catalog_size, k_values):
    """Encode eval_df's users just long enough to fetch embedding rows, then evaluate
    entirely in raw-string-id space (see markdown above) via the shared eval harness."""
    gt = ground_truth_from(eval_df, positive_only=True)          # raw uid -> set(raw item_id)
    users = [uid for uid in gt if uid in uid2u]                  # drop cold (train-unseen) users
    idx_rows = [uid2u[uid] for uid in users]
    uvecs = mfbpr_user_vecs(model, idx_rows)
    index = EmbeddingIndex(export_item_embeddings(model)[:n_items], item_ids)  # UNK row excluded
    recs, metrics = evaluate_index(index, uvecs, users, gt, train_df, catalog_size, k_values)
    return recs, metrics

## Hyperparameter search (val only)

Small grid over `dim` / `lr` / `reg`, each run short (300 steps — just enough to rank
configs, not to converge). **Never touches `test` here.**

In [ ]:
grid = [
    dict(dim=32, lr=0.05, reg=1e-5),
    dict(dim=64, lr=0.05, reg=1e-5),
    dict(dim=64, lr=0.02, reg=1e-4),
]
search_rows = []
for cfg in grid:
    m, _ = train_mfbpr_with_history(train, n_users + 1, n_items + 1, cfg["dim"], cfg["lr"],
                                    cfg["reg"], epochs=1, batch_size=2048,
                                    steps_per_epoch=300, device=DEVICE)
    _, m20 = retrieval_metrics(m, val, train, catalog_size=n_items, k_values=(20,))
    search_rows.append({**cfg, "val_Recall@20": m20["Recall@20"]})

search_df = pd.DataFrame(search_rows).sort_values("val_Recall@20", ascending=False)
search_df

**Pick the best row above** and train the final, longer model with it (more
epochs/steps than the search needed — the search only had to *rank* configs).

In [ ]:
BEST = search_df.iloc[0][["dim", "lr", "reg"]].to_dict()
BEST["dim"] = int(BEST["dim"])
EPOCHS = 20
print("training with:", BEST)

t0 = time.time()
model, loss_history = train_mfbpr_with_history(train, n_users + 1, n_items + 1, BEST["dim"],
                                                BEST["lr"], BEST["reg"], EPOCHS,
                                                batch_size=4096, steps_per_epoch=2000,
                                                device=DEVICE)
print(f"trained in {time.time()-t0:.0f}s")

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(range(1, len(loss_history) + 1), loss_history, color="#3b6fa0", marker="o")
ax.set_xlabel("epoch"); ax.set_ylabel("mean BPR loss"); ax.set_title("MF-BPR training loss")
plt.tight_layout(); plt.show()

## Retrieval evaluation (test)

Same `retrieval_metrics` helper, now on the held-out test split.

In [ ]:
recs, mfbpr_metrics = retrieval_metrics(model, test, train, catalog_size=n_items, k_values=C.K_VALUES)
mfbpr_metrics

In [ ]:
results = baseline_results.copy()
results.loc["MF-BPR"] = {k: v for k, v in mfbpr_metrics.items() if k != "n_users_scored"}
results.round(4)

## Interpretation

Compare MF-BPR's row against Popularity's: MF-BPR personalizes — each user gets a *different*
top-N based on their own history — which should buy back Coverage even where raw Recall@k
is close to or below Popularity's on a very sparse, head-heavy catalog like this one (Popularity's
Recall/NDCG is hard to beat purely because so much test-period engagement concentrates on the
same handful of items). Where MF-BPR should visibly win is **Coverage**: it recommends
different items to different users, so across all users it touches far more of the catalog
than Popularity's one fixed list. This is an accuracy/diversity trade-off worth stating
explicitly in `ANALYSIS.md`, not a simple "MF-BPR wins."

**Defense checklist:** derive `bpr_loss` (`-log sigmoid(s_ui - s_uj) + reg`) line-by-line, and
explain why `sample_triplets`'s collision-resample loop matters — a negative that's secretly a
positive would train the model to push down a score it should be pushing up.

In [ ]:
results.to_csv(DATA / "comparison_results.csv")
print("saved -> data/comparison_results.csv (Notebook 04 appends Two-tower to this)")